<h1 align = "Center">
Lotus Group Customer Behavior Analysis

<h2 align = "Center">
Feature Engineering and Target Construction
</h1>

This is one of the most important phases of the project.

Our two modelling objectives are:

**Classification**: Predict whether a customer will make at least one completed purchase in the next 90 days.

**Regression**: Predict how much a customer will spend on completed purchases in the next 90 days.
An important modelling decision

We don't want to simply use the entire historical customer spending table as our model input.

For example, suppose we want to predict customer behaviour as at 31 December 2023.

Our model must use information available on or before that date. Purchases made in 2024 must be used to establish the prediction targets, not the historical features.

To achieve this, we want to use quarterly observation dates.

For instance:

|Observation date	|Historical features	|Future target period|
|---|---|---|
|31 March 2022	|Data up to 31 March 2022	|Following 90 days|
|30 June 2022	|Data up to 30 June 2022	|Following 90 days|
|30 September 2022	|Data up to 30 September 2022	|Following 90 days|
|...	|...	|...|
|30 June 2024	|Data up to 30 June 2024	|Following 90 days|
|30 September 2024	|Data up to 30 September 2024	|Following 90 days|


* This should give us multiple historical observations per customer rather than relying on just one snapshot. 

* It should also allow us to evaluate the models chronologically.

We want to retain all 3,000 registered customers in each observation period, including customers who have never purchased. 

- Should be important because identifying potential first-time buyers is also commercially valuable.

We'll start by constructing the observation-date framework.

# Establishing Observation Dates

Our dataset covers 1st January 2022 to 31 December 2024.

We want to use quarterly observation dates, beginning on 31st March 2022.

**Why start in March?** Because it gives us an initial three months of transaction history before the first observation date.

Also, we want to stop at 30th September 2024 because we need a complete 90-day period after every observation date. Our dataset does not extend beyond 31st December 2024.

In [1]:
import pandas as pd
# Generate quarterly observation dates
snapshot_dates = pd.date_range(
    start="2022-03-31",
    end="2024-09-30",
    freq="QE"
)

# Convert into a DataFrame

snapshots = pd.DataFrame({
    "snapshot_date": snapshot_dates
})

# Calculate the end of each 90-day prediction window

snapshots["target_end_date"] = (
    snapshots["snapshot_date"] + pd.Timedelta(days=90)
)

snapshots

,snapshot_date,target_end_date
0,2022-03-31,2022-06-29
1,2022-06-30,2022-09-28
2,2022-09-30,2022-12-29
3,2022-12-31,2023-03-31
4,2023-03-31,2023-06-29
5,2023-06-30,2023-09-28
6,2023-09-30,2023-12-29
7,2023-12-31,2024-03-30
8,2024-03-31,2024-06-29
9,2024-06-30,2024-09-28


#### Our observation framework

The final observation date is 30th September 2024, with a target window ending 29th December 2024.

This is valid because the dataset extends to 31st December 2024, giving us the complete 90-day outcome period.

We wnt to use the following convention consistently:
- Historical transactions: `order_date <= snapshot_date`
- Future transactions: `order_date > snapshot_date` and `order_date <= target_end_date`

This prevents information from the future from entering our historical customer features.

### Creating Customer Snapshot Framework

Now we create a dataset containing every registered customer at every observation date.

We have:
- 3,000 registered customers.
- 11 observation dates.
- 33,000 expected customer-observation records.

**Why include all customers?**

We want the models to learn about:
- Existing customers who purchase frequently.
- Customers whose purchasing activity has declined.
- Customers who have never purchased.
- Customers who have registered but have not yet made a completed purchase.

In [2]:
from pathlib import Path

# Define path to data folder
data_path = Path(r"../01_data/processed")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

In [3]:
# Extract customer dataset
customers_clean = load_csv("customers_clean.csv")

# Create every possible customer-observation combination
customer_snapshots = (
    customers_clean.assign(key=1)
    .merge(
        snapshots.assign(key=1),
        on="key"
    )
    .drop(columns="key")
)

# Retain only customers registered by each observation date

customer_snapshots = customer_snapshots[
    customer_snapshots["registration_date"]
    <= customer_snapshots["snapshot_date"]
].copy()

# Sort the records

customer_snapshots = customer_snapshots.sort_values(
    ["snapshot_date", "customer_id"]
).reset_index(drop=True)

# Validate

print("Total records:", len(customer_snapshots))

print(
    "Unique customers:",
    customer_snapshots["customer_id"].nunique()
)

print(
    "Unique observation dates:",
    customer_snapshots["snapshot_date"].nunique()
)

print(
    "Invalid registration dates:",
    (
        customer_snapshots["registration_date"]
        > customer_snapshots["snapshot_date"]
    ).sum()
)

# Inspect the resulting dataset

print("Total records:", len(customer_snapshots))
print("Unique customers:", customer_snapshots["customer_id"].nunique())
print("Unique observation dates:", customer_snapshots["snapshot_date"].nunique())

customer_snapshots.head(10)

Total records: 27548
Unique customers: 3000
Unique observation dates: 11
Invalid registration dates: 0
Total records: 27548
Unique customers: 3000
Unique observation dates: 11


,Unnamed: 0,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,snapshot_date,target_end_date
0,1618,CUS00003,Zeinab Ali,Female,1993-12-26,1289701163,zeinab.ali46@yahoo.com,Zagazig,Nile Delta,Silver,2020-10-21,2022-03-31,2022-06-29
1,636,CUS00008,Ayman Lotfy,Male,1970-12-07,1047358810,NaN,Luxor,Upper Egypt,Bronze,2021-06-11,2022-03-31,2022-06-29
2,1293,CUS00009,Hossam Barakat,Male,NaN,1262686364,hossam.barakat87@hotmail.com,Mansoura,Nile Delta,Bronze,2020-11-05,2022-03-31,2022-06-29
3,2391,CUS00010,Ibrahim Mansour,Male,1993-12-05,1262138657,ibrahim.mansour82@yahoo.com,Tanta,Nile Delta,Bronze,2021-08-10,2022-03-31,2022-06-29
4,2556,CUS00012,Mohamed Farouk,Male,1988-01-09,1082156607,mohamed.farouk64@gmail.com,Ismailia,Canal Zone,Silver,2020-05-25,2022-03-31,2022-06-29
5,1655,CUS00016,Samah Badawi,Female,NaN,1064069715,samah.badawi63@hotmail.com,Cairo,Greater Cairo,Silver,2020-12-03,2022-03-31,2022-06-29
6,1986,CUS00018,Maged Mohamed,Male,NaN,1110090721,maged.mohamed84@hotmail.com,Alexandria,Alexandria,Silver,2021-05-03,2022-03-31,2022-06-29
7,1101,CUS00019,Tarek Khalil,Male,1973-07-21,1141426254,tarek.khalil10@yahoo.com,Giza,Greater Cairo,Bronze,2021-09-22,2022-03-31,2022-06-29
8,3025,CUS00022,Ibrahim Lotfy,Male,1992-07-11,1043257324,ibrahim.lotfy78@hotmail.com,Cairo,Greater Cairo,Bronze,2020-11-10,2022-03-31,2022-06-29
9,534,CUS00023,Amira Shaaban,Female,1988-11-02,1076422705,NaN,Giza,Greater Cairo,Bronze,2020-06-11,2022-03-31,2022-06-29


We now have 27,548 customer-observation records representing 3,000 customers across 11 quarterly observation dates.

Our customer snapshot framework is ready.

## Calculating Historical Purchasing Features

We will begin with four fundamental customer behaviour features:
1. Recency: Days since the customer's last completed purchase.
2. Frequency: Number of completed orders placed before or on the observation date.
3. Monetary: Total amount spent on completed orders before or on the observation date.
4. Average order value (AOV): Average value of the customer's completed orders.

These are commonly known as RFM-related features. They are useful because they summarise how recently, frequently, and how much a customer purchases.

**Important consideration**

We want to calculate these features separately for every observation date.

For example, when calculating a customer's features as at 30th June 2023, we must exclude purchases made after 30th June 2023, even though those transactions exist in our dataset.

### Calculating historical features

We'll start by creating a function that calculates these four features for a given observation date.

In [4]:
# Define completed orders data frame
orders = load_csv("orders_clean.csv")

completed_orders = orders[
    orders["order_status"] == "Completed"
].copy()

def calculate_customer_features(snapshot_date):

    # Retain only completed orders available by the observation date
    historical_orders = completed_orders[
        completed_orders["order_date"] <= snapshot_date
    ].copy()

    # Calculate customer-level historical purchasing behaviour
    customer_features = (
        historical_orders
        .groupby("customer_id")
        .agg(
            last_purchase_date=("order_date", "max"),
            total_orders=("order_id", "nunique"),
            total_spending=("total_revenue", "sum"),
            average_order_value=("total_revenue", "mean")
        )
        .reset_index()
    )

    # Calculate recency
    customer_features["recency_days"] = (
        snapshot_date - customer_features["last_purchase_date"]
    ).dt.days

    # Remove the intermediate date column
    customer_features = customer_features.drop(
        columns="last_purchase_date"
    )

    return customer_features

### Testing the Function

Before we apply it to all 11 observation dates, let's test it using the first observation date.

In [5]:
completed_orders["order_date"] = pd.to_datetime(
    completed_orders["order_date"],
    errors = "coerce"
)

customer_snapshots["snapshot_date"] = pd.to_datetime(
    customer_snapshots["snapshot_date"],
    errors = "coerce"
)

test_features = calculate_customer_features(
    snapshots.loc[0, "snapshot_date"]
)

print("Observation date:", snapshots.loc[0, "snapshot_date"])
print("Customers with completed purchases:",
      test_features["customer_id"].nunique())

test_features.head(10)

Observation date: 2022-03-31 00:00:00
Customers with completed purchases: 733


,customer_id,total_orders,total_spending,average_order_value,recency_days
0,CUS00012,1,930.0,930.00,8
1,CUS00014,2,825.5,412.75,28
2,CUS00016,1,260.0,260.00,41
3,CUS00020,1,160.0,160.00,44
4,CUS00021,1,14656.5,14656.50,51
5,CUS00024,2,2283.5,1141.75,62
6,CUS00027,1,170.0,170.00,84
7,CUS00031,1,27192.0,27192.00,74
8,CUS00033,1,2627.0,2627.00,20
9,CUS00034,1,153.0,153.00,36


733 customers had made at least one completed purchase by 31st March 2022.

We are not discarding the remaining 2,267 customers. 
- They may be new customers, customers who have not purchased yet, or customers whose recorded orders were cancelled or returned.

We want to preserve all of them in the final modelling dataset.

### Applying Historical Features to All Observation Dates

With the function working for the first observation date, let's apply it across all 11 dates.

In [6]:
# Calculate historical purchasing features for every observation date

historical_features = pd.concat(
    [
        calculate_customer_features(snapshot_date)
        .assign(snapshot_date=snapshot_date)
        for snapshot_date in snapshots["snapshot_date"]
    ],
    ignore_index=True
)

print("Total historical feature records:", len(historical_features))
print("Unique customers:", historical_features["customer_id"].nunique())
print("Unique observation dates:", historical_features["snapshot_date"].nunique())

historical_features.head(10)

Total historical feature records: 24353
Unique customers: 2868
Unique observation dates: 11


,customer_id,total_orders,total_spending,average_order_value,recency_days,snapshot_date
0,CUS00012,1,930.0,930.00,8,2022-03-31
1,CUS00014,2,825.5,412.75,28,2022-03-31
2,CUS00016,1,260.0,260.00,41,2022-03-31
3,CUS00020,1,160.0,160.00,44,2022-03-31
4,CUS00021,1,14656.5,14656.50,51,2022-03-31
5,CUS00024,2,2283.5,1141.75,62,2022-03-31
6,CUS00027,1,170.0,170.00,84,2022-03-31
7,CUS00031,1,27192.0,27192.00,74,2022-03-31
8,CUS00033,1,2627.0,2627.00,20,2022-03-31
9,CUS00034,1,153.0,153.00,36,2022-03-31


#### Findings

The output is reasonable.

We have:

|Metric	|Result|
|---|---:|
|Total customer-observation records	|33,000|
|Records with historical completed purchases	|24,353|
|Unique customers with at least one historical completed purchase	|2,868|
|Observation dates	|11|


The difference between 33,000 and 24,353 represents customer-observation periods in which customers had not yet made a completed purchase.

Notice that 2,868 is the number of distinct customers who had purchased at least once by at least one observation date. It does not mean only 2,868 customers were active at every observation date.

### Merging Historical Features Into the Full Customer Snapshots

We want to bring the historical features into customer_snapshots.

We want to use a left join so that no customer-observation record is lost.

In [7]:
# Merge historical purchasing features into the full customer snapshots

modeling_data = customer_snapshots.merge(
    historical_features,
    on=["customer_id", "snapshot_date"],
    how="left",
    validate="one_to_one"
)

print("Total records:", len(modeling_data))
print("Unique customers:", modeling_data["customer_id"].nunique())
print("Unique observation dates:", modeling_data["snapshot_date"].nunique())

modeling_data.head(10)

Total records: 27548
Unique customers: 3000
Unique observation dates: 11


,Unnamed: 0,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,snapshot_date,target_end_date,total_orders,total_spending,average_order_value,recency_days
0,1618,CUS00003,Zeinab Ali,Female,1993-12-26,1289701163,zeinab.ali46@yahoo.com,Zagazig,Nile Delta,Silver,2020-10-21,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
1,636,CUS00008,Ayman Lotfy,Male,1970-12-07,1047358810,NaN,Luxor,Upper Egypt,Bronze,2021-06-11,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
2,1293,CUS00009,Hossam Barakat,Male,NaN,1262686364,hossam.barakat87@hotmail.com,Mansoura,Nile Delta,Bronze,2020-11-05,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
3,2391,CUS00010,Ibrahim Mansour,Male,1993-12-05,1262138657,ibrahim.mansour82@yahoo.com,Tanta,Nile Delta,Bronze,2021-08-10,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
4,2556,CUS00012,Mohamed Farouk,Male,1988-01-09,1082156607,mohamed.farouk64@gmail.com,Ismailia,Canal Zone,Silver,2020-05-25,2022-03-31,2022-06-29,1.0,930.0,930.0,8.0
5,1655,CUS00016,Samah Badawi,Female,NaN,1064069715,samah.badawi63@hotmail.com,Cairo,Greater Cairo,Silver,2020-12-03,2022-03-31,2022-06-29,1.0,260.0,260.0,41.0
6,1986,CUS00018,Maged Mohamed,Male,NaN,1110090721,maged.mohamed84@hotmail.com,Alexandria,Alexandria,Silver,2021-05-03,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
7,1101,CUS00019,Tarek Khalil,Male,1973-07-21,1141426254,tarek.khalil10@yahoo.com,Giza,Greater Cairo,Bronze,2021-09-22,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
8,3025,CUS00022,Ibrahim Lotfy,Male,1992-07-11,1043257324,ibrahim.lotfy78@hotmail.com,Cairo,Greater Cairo,Bronze,2020-11-10,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN
9,534,CUS00023,Amira Shaaban,Female,1988-11-02,1076422705,NaN,Giza,Greater Cairo,Bronze,2020-06-11,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN


#### Handling Customers Without Historical Purchases

For customers who have not made a completed purchase by an observation date:
- total_orders should be 0.
- total_spending should be 0.
- average_order_value should be 0.
- recency_days should remain missing for now.

> A customer who has never purchased does not have a meaningful last purchase date. Assigning zero recency would incorrectly suggest that they purchased today.

In [8]:
# Replace missing historical purchasing counts and amounts with zero

feature_columns = [
    "total_orders",
    "total_spending",
    "average_order_value"
]

modeling_data[feature_columns] = (
    modeling_data[feature_columns].fillna(0)
)

modeling_data.head(10)

,Unnamed: 0,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,snapshot_date,target_end_date,total_orders,total_spending,average_order_value,recency_days
0,1618,CUS00003,Zeinab Ali,Female,1993-12-26,1289701163,zeinab.ali46@yahoo.com,Zagazig,Nile Delta,Silver,2020-10-21,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
1,636,CUS00008,Ayman Lotfy,Male,1970-12-07,1047358810,NaN,Luxor,Upper Egypt,Bronze,2021-06-11,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
2,1293,CUS00009,Hossam Barakat,Male,NaN,1262686364,hossam.barakat87@hotmail.com,Mansoura,Nile Delta,Bronze,2020-11-05,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
3,2391,CUS00010,Ibrahim Mansour,Male,1993-12-05,1262138657,ibrahim.mansour82@yahoo.com,Tanta,Nile Delta,Bronze,2021-08-10,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
4,2556,CUS00012,Mohamed Farouk,Male,1988-01-09,1082156607,mohamed.farouk64@gmail.com,Ismailia,Canal Zone,Silver,2020-05-25,2022-03-31,2022-06-29,1.0,930.0,930.0,8.0
5,1655,CUS00016,Samah Badawi,Female,NaN,1064069715,samah.badawi63@hotmail.com,Cairo,Greater Cairo,Silver,2020-12-03,2022-03-31,2022-06-29,1.0,260.0,260.0,41.0
6,1986,CUS00018,Maged Mohamed,Male,NaN,1110090721,maged.mohamed84@hotmail.com,Alexandria,Alexandria,Silver,2021-05-03,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
7,1101,CUS00019,Tarek Khalil,Male,1973-07-21,1141426254,tarek.khalil10@yahoo.com,Giza,Greater Cairo,Bronze,2021-09-22,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
8,3025,CUS00022,Ibrahim Lotfy,Male,1992-07-11,1043257324,ibrahim.lotfy78@hotmail.com,Cairo,Greater Cairo,Bronze,2020-11-10,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN
9,534,CUS00023,Amira Shaaban,Female,1988-11-02,1076422705,NaN,Giza,Greater Cairo,Bronze,2020-06-11,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN


#### Result Validation

In [9]:
print("Total modelling records:", len(modeling_data))

print("Missing values in historical features:")
print(modeling_data[
    ["recency_days", "total_orders", "total_spending", "average_order_value"]
].isna().sum())

print("\nCustomers with no historical completed purchases:")
print((modeling_data["total_orders"] == 0).sum())

Total modelling records: 27548
Missing values in historical features:
recency_days           6098
total_orders              0
total_spending            0
average_order_value       0
dtype: int64

Customers with no historical completed purchases:
6098


#### **What the output tells us**:
* 27,548 modelling records: These are eligible customer–snapshot observations after excluding customers who had not yet registered.
* 6,098 observations without previous completed purchases: These are registered customers who had not made a completed purchase by the relevant snapshot date.
* 6,098 missing `recency_days` values: This is expected. A customer with no previous completed purchase has no last-purchase date from which to calculate recency.
* Zero missing values in the other historical features: Filling total_orders, total_spending, and `average_order_value` with zero was appropriate.

We want to retain the missing `recency_days` values for now. Later, when preparing the modelling data, we can handle them appropriately.

### Adding Customer Tenure

Customer tenure measures how long a customer has been registered with Lotus Group as of the observation date.

- It is useful because we wouldn't want to judge a customer registered two weeks ago by the same purchasing expectations as someone registered two years ago.

In [10]:
# Parse snapshot_date and registration_date
date_cols = ["snapshot_date","registration_date"]
modeling_data[date_cols] = modeling_data[date_cols].apply(pd.to_datetime,
    errors = "coerce"
)

modeling_data["customer_tenure_days"] = (
    modeling_data["snapshot_date"]
    - modeling_data["registration_date"]
).dt.days

In [11]:
# Inspect results
modeling_data["customer_tenure_days"].describe()

count    27548.000000
mean       690.223610
std        412.568497
min          2.000000
25%        350.000000
50%        657.000000
75%        989.000000
max       1734.000000
Name: customer_tenure_days, dtype: float64

In [12]:
# Verify no negative values exist
(modeling_data["customer_tenure_days"] < 0).sum()

np.int64(0)

#### Implication

Tenure feature has passed validation:
- 27,548 observations have tenure values.
- Average customer tenure: approximately 690 days (1.9 years).
- Median tenure: 657 days (1.8 years).
- Minimum tenure: 2 days.
- Maximum tenure: 1,734 days.
- Negative tenure values: 0.

Feature is valid and can be retained.

## Constructing Prediction Targets

We want to create the two outcomes our models will predict.

For every customer–snapshot observation:
1. Classification target (will_purchase): Did the customer make at least one completed purchase during the following 90 days?
   - 1 = Yes
   - 0 = No
2. Regression target (future_spending): How much did the customer spend on completed orders during those same 90 days?
   - Amount in EGP.
   - Customers who made no completed purchases receive 0.

We want to use completed_orders, which already contains only completed orders.

### Function to calculate future outcomes

In [13]:
def calculate_future_targets(snapshot_date, target_end_date):

    future_orders = completed_orders[
        (completed_orders["order_date"] > snapshot_date) &
        (completed_orders["order_date"] <= target_end_date)
    ].copy()

    customer_targets = (
        future_orders
        .groupby("customer_id")
        .agg(
            future_orders=("order_id", "nunique"),
            future_spending=("total_revenue", "sum")
        )
        .reset_index()
    )

    customer_targets["will_purchase"] = (
        customer_targets["future_orders"] > 0
    ).astype(int)

    return customer_targets

**Why we use these date conditions**
- order_date > snapshot_date ensures we do not use purchases that happened on or before the observation date.
- order_date <= target_end_date restricts the outcome to the next 90 days.
- Grouping by customer gives us their future purchasing activity and spending.

### Target Generation for all observation dates

In [14]:
future_targets = pd.concat(
    [
        calculate_future_targets(snapshot_date, target_end_date)
        .assign(
            snapshot_date=snapshot_date,
            target_end_date=target_end_date
        )
        for snapshot_date, target_end_date
        in zip(snapshots["snapshot_date"], snapshots["target_end_date"])
    ],
    ignore_index=True
)

> Customers who make no purchases in a future window will not appear in future_targets because the aggregation only includes customers with qualifying orders. We will account for those customers when merging the targets into modeling_data.

In [15]:
# Inspect Results
print("Future target records:", len(future_targets))
print("Unique customers:", future_targets["customer_id"].nunique())
print("Unique snapshot dates:", future_targets["snapshot_date"].nunique())

future_targets.head()

Future target records: 8213
Unique customers: 2873
Unique snapshot dates: 11


,customer_id,future_orders,future_spending,will_purchase,snapshot_date,target_end_date
0,CUS00012,1,1020.5,1,2022-03-31,2022-06-29
1,CUS00013,1,795.0,1,2022-03-31,2022-06-29
2,CUS00017,1,505.0,1,2022-03-31,2022-06-29
3,CUS00021,1,2886.0,1,2022-03-31,2022-06-29
4,CUS00024,1,415.0,1,2022-03-31,2022-06-29


What the output tells us:
- 8,213 customer–snapshot observations have at least one completed purchase in their following 90-day window.
- 2,873 distinct customers made at least one qualifying purchase across the observation periods.
- All 11 observation dates are represented.

> These are ony positive purchasing observations. Customers who made no qualifying purchase are not yet included in future_targets. 

>We need to include them as zeros when merging.

## Merging future targets into the modelling dataset

In [16]:
modeling_data = modeling_data.merge(
    future_targets[
        [
            "customer_id",
            "snapshot_date",
            "future_orders",
            "future_spending",
            "will_purchase"
        ]
    ],
    on=["customer_id", "snapshot_date"],
    how="left",
    validate="one_to_one"
)

In [17]:
# Replace missing target values with zero
target_columns = [
    "future_orders",
    "future_spending",
    "will_purchase"
]

modeling_data[target_columns] = (
    modeling_data[target_columns].fillna(0)
)

A missing match means the customer had no completed purchase during that observation's 90-day target window.

## Validating Targets

In [18]:
# Validate Targets
print("Total modelling records:", len(modeling_data))

print("\nClassification target distribution:")
print(modeling_data["will_purchase"].value_counts())

print("\nClassification target percentages:")
print(
    modeling_data["will_purchase"]
    .value_counts(normalize=True)
    .mul(100)
)

print("\nFuture spending summary:")
print(modeling_data["future_spending"].describe())

print("\nMissing target values:")
print(
    modeling_data[
        ["future_orders", "future_spending", "will_purchase"]
    ].isna().sum()
)

Total modelling records: 27548

Classification target distribution:
will_purchase
0.0    20678
1.0     6870
Name: count, dtype: int64

Classification target percentages:
will_purchase
0.0    75.06171
1.0    24.93829
Name: proportion, dtype: float64

Future spending summary:
count    27548.000000
mean      1082.375526
std       4833.124465
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      83304.500000
Name: future_spending, dtype: float64

Missing target values:
future_orders      0
future_spending    0
will_purchase      0
dtype: int64


We have successfully constructed both prediction targets, and the results reveal something important about the purchasing behaviour in our modelling dataset.

1. Classification target interpretation

|Outcome	|Observations	|Percentage|
|---|---:|---:|
|No qualifying purchase (0)	|20,678	|75.06%|
|Qualifying purchase (1)	|6,870	|24.94%|
|Total	|27,548	|100%|

Approximately one in four customer–snapshot observations results in a completed purchase during the following 90 days.

This is a moderately imbalanced classification problem: non-purchasing observations account for 75% of the data.

**Why it matters**

A model that simply predicts that nobody will purchase would achieve approximately 75% accuracy while being completely useless at identifying future purchasers.

Consequently, we will not rely on accuracy alone when evaluating the classification model. We will pay particular attention to precision, recall and F1-score, alongside ROC-AUC.

2. Regression target interpretation

Our future spending summary shows:

|Statistic	|Future spending (EGP)|
|---|---|
|Mean	|1,082.38|
|Standard deviation	|4,833.12|
|Minimum	|0|
|25th percentile	|0|
|Median	|0|
|75th percentile	|0|
|Maximum	|83,304.50|

The most important observation is that at least 75% of customer–snapshot observations have zero future spending.

This is expected because customers who do not purchase during the 90-day window receive zero spending.

It also means the spending distribution is highly right-skewed: a relatively small group of customers accounts for most future spending.

This presents a challenge for regression modelling. We will account for it when evaluating prediction errors.

#### Validating Relationship Between the Two Targets

In [19]:
# Confirming whether the targets are logically consistent
print("Classification and regression consistency checks")

print(
    "Purchasers with zero future spending:",
    (
        (modeling_data["will_purchase"] == 1) &
        (modeling_data["future_spending"] == 0)
    ).sum()
)

print(
    "Non-purchasers with positive future spending:",
    (
        (modeling_data["will_purchase"] == 0) &
        (modeling_data["future_spending"] > 0)
    ).sum()
)

print(
    "Observations with future orders but no purchase:",
    (
        (modeling_data["future_orders"] > 0) &
        (modeling_data["will_purchase"] == 0)
    ).sum()
)

Classification and regression consistency checks
Purchasers with zero future spending: 0
Non-purchasers with positive future spending: 0
Observations with future orders but no purchase: 0
